In [11]:
print('Ritu')

Ritu


In [12]:
count = 30

In [13]:
from langchain_ai21.chat_models import ChatAI21
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage,ToolMessage
from langgraph.types import interrupt,Command 
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from typing import TypedDict,Annotated, List, Dict, Literal,Optional
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool
from pydantic import BaseModel
from json_repair import repair_json
import re
from typing import List
from dotenv import load_dotenv
import json
import os
load_dotenv()
DB_URL = os.getenv("PPT_URL")

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    action: Literal[ "continue_slide", "update_outline", "update_slide", "complete", '' ]
    tool_caller: Literal["generate_outline","generate_slide_detail"]
class OutlineSlide(BaseModel):
    slide_number: int
    slide_title: str
class OutlineOutput(BaseModel):
    title: str
    total_slides: int
    slides: List[OutlineSlide]
class DetailedPoint(BaseModel):
    key_point: str
    explanation: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    layout: Literal[
        "bullets",
        "bullets_with_text",
        "paragraph",
        "two_column",
        "mixed"
    ]
    intro_line: Optional[str] = None
    bullet_points: Optional[List[str]] = None
    supporting_text: Optional[str] = None
    paragraphs: Optional[List[str]] = None

model = ChatGroq(model="llama-3.3-70b-versatile")

searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
outline_parser = PydanticOutputParser(pydantic_object= OutlineOutput)
# outline_parser = PydanticOutputParser(pydantic_object= OutlineSlide)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)
OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content=f"""You are an expert presentation designer.

Your task:
Generate ONLY the presentation title and slide titles.
    
{outline_parser.get_format_instructions()} 

Strict Rules:
- Generate EXACTLY the number of slides requested by the user.
- Do NOT generate key points.
- Do NOT generate slide content.
- Only generate slide_number and slide_title.
- Slide numbers must start from 1 and increment sequentially.
- Ensure logical flow from introduction to conclusion.
- Keep slide titles concise but descriptive.
- Use tools only if factual accuracy is required.
- Return ONLY valid JSON.
- No markdown.
- No explanations.
""")


DETAIL_SYSTEM_PROMPT = SystemMessage(
content=f"""
You are a professional presentation designer.

Your job is to generate visually balanced slide content for a PowerPoint presentation.

For each slide choose the most appropriate layout or a mix of layouts  and generate structured content.

Available layouts:
- bullets
- bullets_with_text
- paragraph
- mixed


{detailed_parser.get_format_instructions()}

General Principles:
Slides should be informative but not crowded.
A good slide often combines short explanations with bullet points.

Content Elements:
Slides may contain combination of:
- intro_line (a introductory sentence)
- bullet_points (key ideas)
- supporting_text (a short insight or explanation)
- paragraphs (short explanation text)


Layout Guidelines:

bullets
- 6-8 bullet points
- each bullet 6–12 words
- may optionally include a intro_line before bullets
- Some bullet slides should include a short explanation sentence before or after the bullets.

bullets_with_text
- 5-8 bullet points
- include supporting_text (1 short explanation sentence)
- may optionally include intro_line
- Some bullet slides should include a short explanation sentence before or after the bullets.

paragraph
- 3-5 short paragraphs
- each paragraph max 40 words
- optionally include a short intro_line

Slides may contain a combination of:

- intro_line (1 explanation sentence)
- bullet_points (3–5 bullets)
- paragraphs (1 short paragraph)
- supporting_text (1 insight sentence)

Good slides often combine elements.
Example structure:

intro_line
bullet_points
supporting_text


Layout Distribution Rules:

- 40–50% slides → bullets
- 20–30% slides → bullets_with_text
- 10–20% slides → paragraph

Variation Rules:
Do NOT generate the same layout repeatedly.
Use different content styles across slides.


Content Density Rules:
Slides should contain roughly 80-120 words total.
Content must fit comfortably on a PowerPoint slide.

Quality Rules:
- Bullet points should be concise and informative.
- Avoid repeating the same wording.
- Ensure the slide content is clear when presented visually.
Return JSON only.
"""
)


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]
    result = model_with_tools.invoke(messages)
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 
    if result.content:
        try:
            output['outline'] = outline_parser.parse(repair_json(str(result.content))).model_dump()
            print("output['outline']",output['outline'])
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides',[])
    total_slides = len(state.get('outline',{}).get('slides',[]))
    current_slide = outline['slides'][int(current_index)]
    if current_index >= total_slides or current_slide['slide_title'] is None:
        return {"action": "complete"}
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index":current_index
    }
    if state['action'] == "update_slide":
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        last_outline = outline['slides'][current_index-1]
        output['feedback'] = ''
        output['action'] = ''
        prompt = HumanMessage(
            content=f"""
Update this slide content.

Presentation Title:
{outline['title']}

Outline of the slide:
{last_outline}

Current Slide Content:
{last_slide}

User Feedback:
{feedback}
"""
        )

    else:
        
        
        prompt = HumanMessage(
            content=f"""Generate detailede content for this slide:
Presentation Title: {outline['title']}
Slide Title: {current_slide['slide_title']}
Slide Number: {current_slide['slide_number']}

Provide comprehensive, presentation-ready content."""
    )
    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    if isinstance( state['messages'][-1],ToolMessage):
        messages = state['messages'][-2:]+[DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    if result.content:
        try:
            detailed_slides.append(detailed_parser.parse(repair_json(str(result.content))).model_dump())
            output['detailed_slides'] = detailed_slides 
            output['current_slide_index'] = current_index +1
            
        except json.JSONDecodeError as e:
            print('generate_slide_detail_node inside',e)
    if output['current_slide_index'] == total_slides:
        output["action"] =  "complete"
    output['messages'] = [result]
    return output
def route_after_tools(state: PptState):
    return state["tool_caller"]
def human_decision(state: PptState):
    decision = interrupt({})
    if decision['action'] == "update_outline":

        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}
    elif decision['action'] == 'update_slide':
        return {'action':'update_slide'}
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"
    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    elif action == 'complete':  
        return END
    return END
def build_workflow():
    workflow = StateGraph(PptState)
    workflow.add_node("generate_outline", generate_outline_node)
    workflow.add_node("generate_slide_detail", generate_slide_detail_node)
    workflow.add_node("human_decision", human_decision)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_edge(START, "generate_outline")
    workflow.add_conditional_edges(
        "generate_outline",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "generate_slide_detail",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "tools",
        route_after_tools,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
        },
    )
    workflow.add_conditional_edges(
        "human_decision",
        route_after_human,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
            END: END,
        },
    )
    return workflow
def create_ckeckpointer_and_graph(db_url: str):
    if not db_url:
        raise ValueError('Database Url environment variable not set')
    connection_kwargs = {
            "autocommit": True,
            "prepare_threshold": 0,
        }
    pool = ConnectionPool(
        conninfo=db_url,
            max_size=20,
            kwargs=connection_kwargs,
    )
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    workflow = build_workflow()
    graph = workflow.compile(checkpointer=checkpointer)
    return checkpointer, graph


In [16]:
import random

topics = ["Impact of Social Media on Human Relationships",
"Plastic Pollution and Its Effect on the Environment",
"Importance of Mental Health in Modern Life",
"Urbanization and Its Impact on Cities",
"Healthy Lifestyle and the Role of Exercise",
"Electric Vehicles and the Future of Transportation",
"The Importance of Time Management for Students",
"Global Warming: Causes and Solutions",
"Women Empowerment in Modern Society",
"The Role of Education in Personal Development",]


checkpointer, graph = create_ckeckpointer_and_graph(DB_URL)

topic = random.choice(topics)
num_slides = 10
prompt = HumanMessage(
    content=f"""
Create EXACTLY {num_slides} slide titles for a presentation on:

Topic: {topic}

Do not create fewer or more slides.
"""
)
config = {'configurable':{'thread_id':'15-03-26-2'}}
state = {
            "messages": [prompt],
            "topic":topic,
            "outline": {},
            "detailed_slides": [],
            "current_slide_index": 0,
            "feedback": "",
            "action": "",
            "tool_caller": "generate_outline",
        }
result = graph.invoke(state,config = config)

output['outline'] {'title': 'Impact of Social Media on Human Relationships', 'total_slides': 10, 'slides': [{'slide_number': 1, 'slide_title': 'Introduction to Social Media'}, {'slide_number': 2, 'slide_title': 'The Rise of Social Media'}, {'slide_number': 3, 'slide_title': 'Changing Communication Dynamics'}, {'slide_number': 4, 'slide_title': 'Impact on Mental Health'}, {'slide_number': 5, 'slide_title': 'Social Media and Relationships'}, {'slide_number': 6, 'slide_title': 'The Concept of Online Identity'}, {'slide_number': 7, 'slide_title': 'Cyberbullying and Harassment'}, {'slide_number': 8, 'slide_title': 'Social Comparison and Self-Esteem'}, {'slide_number': 9, 'slide_title': 'Rebuilding Genuine Connections'}, {'slide_number': 10, 'slide_title': 'Conclusion and Future Directions'}]}


In [17]:
for i in range(10):
    state = Command(resume={
        "action":'continue_slide'
    })
    result1 = graph.invoke(state,config = config)
    print(result1)

{'messages': [HumanMessage(content='\nCreate EXACTLY 10 slide titles for a presentation on:\n\nTopic: Impact of Social Media on Human Relationships\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='7bd1bb5a-b0da-403d-b631-2706a13968d6'), AIMessage(content='{"title": "Impact of Social Media on Human Relationships", "total_slides": 10, "slides": [{"slide_number": 1, "slide_title": "Introduction to Social Media"}, {"slide_number": 2, "slide_title": "The Rise of Social Media"}, {"slide_number": 3, "slide_title": "Changing Communication Dynamics"}, {"slide_number": 4, "slide_title": "Impact on Mental Health"}, {"slide_number": 5, "slide_title": "Social Media and Relationships"}, {"slide_number": 6, "slide_title": "The Concept of Online Identity"}, {"slide_number": 7, "slide_title": "Cyberbullying and Harassment"}, {"slide_number": 8, "slide_title": "Social Comparison and Self-Esteem"}, {"slide_number": 9, "slide_title": "Rebuilding Genuine Connection

In [18]:
result1

{'messages': [HumanMessage(content='\nCreate EXACTLY 10 slide titles for a presentation on:\n\nTopic: Impact of Social Media on Human Relationships\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='7bd1bb5a-b0da-403d-b631-2706a13968d6'),
  AIMessage(content='{"title": "Impact of Social Media on Human Relationships", "total_slides": 10, "slides": [{"slide_number": 1, "slide_title": "Introduction to Social Media"}, {"slide_number": 2, "slide_title": "The Rise of Social Media"}, {"slide_number": 3, "slide_title": "Changing Communication Dynamics"}, {"slide_number": 4, "slide_title": "Impact on Mental Health"}, {"slide_number": 5, "slide_title": "Social Media and Relationships"}, {"slide_number": 6, "slide_title": "The Concept of Online Identity"}, {"slide_number": 7, "slide_title": "Cyberbullying and Harassment"}, {"slide_number": 8, "slide_title": "Social Comparison and Self-Esteem"}, {"slide_number": 9, "slide_title": "Rebuilding Genuine Connecti

In [49]:
from pptx import Presentation
from pptx.util import Pt

prs = Presentation()

def set_font_size(text_frame, size):
    for paragraph in text_frame.paragraphs:
        for run in paragraph.runs:
            run.font.size = Pt(size)

for slide_data in result1["detailed_slides"]:

    layout_type = slide_data.get("layout")

    if layout_type == "title_only":
        slide = prs.slides.add_slide(prs.slide_layouts[5])
    else:
        slide = prs.slides.add_slide(prs.slide_layouts[1])

    # ---------- TITLE ----------
    title = slide.shapes.title
    title.text = slide_data["slide_title"]

    title_size = 42 if layout_type == "title_only" else 32

    for p in title.text_frame.paragraphs:
        for r in p.runs:
            r.font.size = Pt(title_size)

    if layout_type == "title_only":
        continue

    body = slide.placeholders[1].text_frame
    body.clear()

    # ---------- INTRO LINE ----------
    intro = slide_data.get("intro_line")

    if intro:
        body.text = intro
        for r in body.paragraphs[0].runs:
            r.font.size = Pt(20)

    # ---------- BULLETS ----------
    bullets = slide_data.get("bullet_points", [])

    if bullets:

        if not intro:
            body.text = bullets[0]
            bullets = bullets[1:]
        else:
            p = body.add_paragraph()
            p.text = bullets[0]
            p.level = 0
            bullets = bullets[1:]

        for bullet in bullets:
            p = body.add_paragraph()
            p.text = bullet
            p.level = 0

    # ---------- PARAGRAPHS ----------
    paragraphs = slide_data.get("paragraphs")

    if paragraphs:
        for para in paragraphs:
            p = body.add_paragraph()
            p.text = para
            p.level = 0

    # ---------- SUPPORTING TEXT ----------
    supporting = slide_data.get("supporting_text")

    if supporting:
        p = body.add_paragraph()
        p.text = supporting
        p.level = 1

        for r in p.runs:
            r.font.size = Pt(18)

    # ---------- GENERAL FONT CONTROL ----------
    set_font_size(body, 20)

prs.save(f"presentation{count}.pptx")
count+=1

In [19]:
import os
import random
from pptx import Presentation
from pptx.util import Pt, Inches


def create_presentation_from_data(data, output_name="plastic_pollution.pptx"):
    """
    Create a PowerPoint presentation from structured data with optimized font sizes.

    Args:
        data: List of dictionaries containing slide information
        output_name: Name of the output file

    Returns:
        Path to the created presentation
    """

    # ============ FONT SIZE CONFIGURATION ============
    # Adjust these values to your preference
    FONT_SIZES = {
        'title': 44,           # Slide title (larger, bold)
        'intro': 20,           # Introduction/overview text
        'bullets': 18,         # Main bullet points
        'sub_bullets': 16,     # Sub-bullet points (if using level 2)
        'supporting': 16,      # Supporting/summary text
        'paragraphs': 16,      # Additional paragraphs
        'small_text': 14       # Footer or fine print
    }

    # ============ LOAD TEMPLATE ============
    template_folder = "../templates"

    if os.path.exists(template_folder):
        print('Template folder found')
        templates = [f for f in os.listdir(template_folder) if f.endswith(".pptx")]
        print(f'Available templates: {templates}')

        if templates:
            # selected_template = 'designe5.pptx'
            selected_template = random.choice(templates)
            print(f"Using template: {selected_template}")
            prs = Presentation(os.path.join(template_folder, selected_template))
        else:
            print("No templates found. Creating presentation with default theme.")
            prs = Presentation()
    else:
        print("Template folder not found. Creating presentation with default theme.")
        prs = Presentation()

    # Get available slide layouts
    available_layouts = prs.slide_layouts
    print(f"Available layouts: {len(available_layouts)}\n")

    # ============ CREATE SLIDES ============
    for slide_data in data:
        # Select layout
        if len(available_layouts) > 1:
            layout = available_layouts[1]  # Title and Content layout
        else:
            layout = available_layouts[0]

        slide = prs.slides.add_slide(layout)

        # ============ SET TITLE ============
        if slide.shapes.title:
            title_text = slide_data.get("slide_title", "")
            slide.shapes.title.text = title_text

            # Set title font size
            title_frame = slide.shapes.title.text_frame
            for paragraph in title_frame.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(FONT_SIZES['title'])
                    run.font.bold = True

        # ============ FIND BODY PLACEHOLDER ============
        body_shape = None
        for shape in slide.placeholders:
            if shape.placeholder_format.type == 2:  # BODY type
                body_shape = shape
                break

        # Fallback to index 1 if type search fails
        if not body_shape:
            try:
                body_shape = slide.placeholders[1]
            except (KeyError, IndexError):
                print(f"⚠️  Warning: No body placeholder found for slide {slide_data.get('slide_number')}")
                continue

        text_frame = body_shape.text_frame
        text_frame.clear()
        text_frame.word_wrap = True

        # ============ ADD INTRO LINE ============
        intro = slide_data.get("intro_line")
        if intro:
            p = text_frame.paragraphs[0]
            p.text = intro
            p.level = 0
            p.space_after = Pt(12)  # Add spacing after intro

            # Set intro font size (larger for emphasis)
            if p.runs:
                for run in p.runs:
                    run.font.size = Pt(FONT_SIZES['intro'])
                    run.font.bold = False

        # ============ ADD BULLET POINTS ============
        bullets = slide_data.get("bullet_points", [])
        for bullet in bullets:
            p = text_frame.add_paragraph()
            p.text = bullet
            p.level = 1  # First level bullets
            p.space_after = Pt(8)  # Spacing between bullets

            # Set bullet font size
            if p.runs:
                for run in p.runs:
                    run.font.size = Pt(FONT_SIZES['bullets'])

        # ============ ADD SUPPORTING TEXT ============
        supporting = slide_data.get("supporting_text")
        if supporting:
            # Add spacing before supporting text
            p = text_frame.add_paragraph()
            p.text = ""
            p.level = 0
            p.space_after = Pt(6)

            # Add supporting text
            p = text_frame.add_paragraph()
            p.text = supporting
            p.level = 0
            p.space_after = Pt(8)

            # Set supporting text font (smaller, italic)
            if p.runs:
                for run in p.runs:
                    run.font.size = Pt(FONT_SIZES['supporting'])
                    run.font.italic = True

        # ============ ADD PARAGRAPHS ============
        paragraphs = slide_data.get("paragraphs", [])
        if paragraphs:
            # Add spacing before paragraphs
            p = text_frame.add_paragraph()
            p.text = ""
            p.level = 0
            p.space_after = Pt(6)

            for para in paragraphs:
                p = text_frame.add_paragraph()
                p.text = para
                p.level = 0
                p.space_after = Pt(8)

                # Set paragraph font size
                if p.runs:
                    for run in p.runs:
                        run.font.size = Pt(FONT_SIZES['paragraphs'])

        print(f"✓ Slide {slide_data.get('slide_number')}: {slide_data.get('slide_title')}")

    # ============ SAVE PRESENTATION ============
    prs.save(output_name)
    print(f"\n✅ PPT Generated Successfully: {output_name}")
    print(f"\n📊 Font Sizes Used:")
    print(f"   • Title: {FONT_SIZES['title']}pt (Bold)")
    print(f"   • Intro: {FONT_SIZES['intro']}pt")
    print(f"   • Bullets: {FONT_SIZES['bullets']}pt")
    print(f"   • Supporting Text: {FONT_SIZES['supporting']}pt (Italic)")
    print(f"   • Paragraphs: {FONT_SIZES['paragraphs']}pt")

    return output_name


# Example usage
if __name__ == "__main__":
    # Your data here (same structure as before)
    

    # Create presentation with counter
    # count = 1  # Initialize counter
    output_file = create_presentation_from_data(
        result1["detailed_slides"], 
        output_name=f"plastic_pollution_presentation_{count}.pptx"
    )
    count += 1


Template folder found
Available templates: ['designe1.pptx', 'designe10.pptx', 'designe11.pptx', 'designe12.pptx', 'designe2.pptx', 'designe3.pptx', 'designe4.pptx', 'designe5.pptx', 'designe6.pptx', 'designe7.pptx', 'designe8.pptx', 'designe9.pptx']
Using template: designe1.pptx
Available layouts: 17

✓ Slide 1: Introduction to Social Media
✓ Slide 2: The Rise of Social Media
✓ Slide 3: Changing Communication Dynamics
✓ Slide 4: Impact on Mental Health
✓ Slide 5: Social Media and Relationships
✓ Slide 6: The Concept of Online Identity
✓ Slide 7: Cyberbullying and Harassment
✓ Slide 8: Social Comparison and Self-Esteem
✓ Slide 9: Rebuilding Genuine Connections
✓ Slide 10: Conclusion and Future Directions

✅ PPT Generated Successfully: plastic_pollution_presentation_30.pptx

📊 Font Sizes Used:
   • Title: 44pt (Bold)
   • Intro: 20pt
   • Bullets: 18pt
   • Supporting Text: 16pt (Italic)
   • Paragraphs: 16pt
